## **Промежуточная контрольная работа 1.**

Загрузите файл "orders_project_01.csv". Этот файл содержит сведения о заказах в интернет магазине.


---



*   **InvoiceNo** - номер заказа
*   **CustomerID** - уникальный идентификатор клиента (может отсутствовать)
*   **Description** -  описание товара
*   **Quantity** - количество товара (если отсутствует то принять равным 1, если некорректна такую запись нужно исключить из рассмотрения)
*   **UnitPrice** - цена за единицу (если отсутствует или меньше или равно 0 такую запись исключить из рассмотрения)
*   **Category** - категория товара
*   **Discount** - скидка (в пределах от 0.0 до 0.3, если отсутствует то принять равной 0.0, все остальное исключить из рассмотрения)
*   **PaymentMethod** - способ оплаты (может быть ошибка в регистре)


---





#### **Task 1.** Получение, очистка и предварительная обработка данных.


1.   Считайте данные из файла
2.   Выполните предварительную очистку данных. Удалите записи с некорректным количеством товара (отрицательное, или не число), удалите записи с некорректной стоимостью, удалите записи с некорректной скидкой.
3.   Заполните пропущенные значения количества и скидки.
4.   Привести все способы оплаты к единому виду (например все в верхнем или нижнем регистре)
5.   Разбейте заказы на две группы. В первой заказы в которой указаны идентификаторы пользователя, во второй нет.


---




In [1]:
from google.colab import drive
drive.mount('/content/drive')
import csv

orders = []

with open('/content/drive/MyDrive/orders_project_01.csv', 'r', encoding='utf-8')as f:
    reader = csv.DictReader(f)
    for order in reader:
        if order["Quantity"] == "":
            quantity = 1
        else:
            try:
                quantity = float(order["Quantity"])
            except:
                continue

            if quantity < 0:
                continue

        if order["UnitPrice"] == "":
            continue

            try:
                price = float(order["UnitPrice"])
            except:
                continue

            if price <= 0:
                continue

        if order["Discount"] == "":
            discount = 0.0

        else:
            try:
                discount = float(order["Discount"])
            except:
                continue

        if discount < 0 or discount > 0.3:
            continue

        order["Quantity"] = quantity
        order["Discount"] = discount

        order["PaymentMethod"] = order["PaymentMethod"].lower()

        orders.append(order)

orders_with_customer = []
orders_without_customer = []

for order in orders:
    if order["CustomerID"]:
        orders_with_customer.append(order)
    else:
        orders_without_customer.append(order)

print("Total number of correct orders:", len(orders))
print("Orders with CustomerID:", len(orders_with_customer))
print("Orders without CustomerID:", len(orders_without_customer))





Mounted at /content/drive
Total number of correct orders: 952
Orders with CustomerID: 450
Orders without CustomerID: 502


#### **Task 2.** Подготовка к анализу данных.
1. Напишите функцию ```group_by_category(orders)``` которая в качестве параметра принимает список заказов и вернет словарь в котором заказы будут сгруппированы по ```Category```.

In [9]:
def group_by_category(orders: list) -> dict[str:list]:
    result = {}

    for order in orders:
        category = order["Category"]

        if category not in result:
            result[category] = []

        result[category].append(order)

    return result

# Закомментировала строку, чтобы не перегружать экран огромным выводом
#print(group_by_category(orders))


2.  Напишите функцию которая определит самый популярный товар (по количеству) и вернет название этого товара.

In [3]:
def most_popular_item(orders: list) -> str:
    products = {}

    for order in orders:
        product = order["Description"]
        quantity = order["Quantity"]

        if product not in products:
            products[product] = 0

        products[product] += quantity

    popular_product = max(products, key=products.get)

    return popular_product

print("The most popular product:", most_popular_item(orders))


The most popular product: Laptop Stand


3.   Напишите функцию ```total_quantity(orders)``` которая в качестве параметра принимает список заказов и вернет общее количество проданных товаров.

In [4]:
def total_quantity(orders: list) -> int:
    total = 0

    for order in orders:
        total += int(order["Quantity"])

    return total

print(total_quantity(orders))

5218


4.   Напишите функцию ```total_sales(orders)``` которая в качестве параметра принимает список заказов и возвращает общую сумму продаж (с учетом скидок).

In [5]:
def total_sales(orders: list) -> float:
    total = 0

    for order in orders:
        total += int(order["Quantity"]) * float(order["UnitPrice"]) * (1 - float(order["Discount"]))

    return total

print(total_sales(orders))

227655.1934


5.   Напишите функцию ```top_customer(orders)``` которая в качестве параметра принимает список заказов и вернет ```CustomerID``` клиента который совершил самый дорогой заказ.


In [6]:
def top_customer(orders: list) -> str:
    max_price = 0
    best_customer = ""

    for order in orders:
        price = int(order["Quantity"]) * float(order["UnitPrice"]) * (1 - float(order["Discount"]))

        if price > max_price:
            max_price = price
            best_customer = order["CustomerID"]

    print(f"The amount of the largest order: {max_price:.2f}")

    return best_customer

print(top_customer(orders))

The amount of the largest order: 985.78
72471533-22fe-4ed0-b676-e32ff6d1605c


6.   Напишите функцию ```top_category(orders)``` которая в качестве параметра принимает список заказов и возвращает категорию с максимальными продажами (по сумме заказов с учетом скидки).

In [7]:
def top_category(orders: list) -> str:
    sales_cat = {}

    for order in orders:
        category = order["Category"]
        price = int(order["Quantity"]) * float(order["UnitPrice"]) * (1 - float(order["Discount"]))

        if category not in sales_cat:
            sales_cat[category] = 0.0
        sales_cat[category] += price

    best_cat = ""
    max_sales = 0

    for cat, sales in sales_cat.items():
        if sales > max_sales:
            max_sales = sales
            best_cat = cat

    return best_cat

print(top_category(orders))



Electronics


7.   Напишите функцию ```sum_by_category(orders)``` которая в качестве параметра принимает список заказов и возвращает словарь где ключ - название категории, значение - сумма все заказов в этой категории (с учетом скидок).


In [55]:
def sum_by_category(orders: list) -> dict[str:float]:
    result = {}

    for order in orders:
        category = order["Category"]
        quantity = order["Quantity"]
        price = float(order["UnitPrice"])
        discount = order["Discount"]

        total = quantity * price
        total = total * (1 - discount)

        if category not in result:
            result[category] = 0

        result[category] += total

    return result

print(sum_by_category(orders))

{'Electronics': 90887.86990000005, 'Accessories': 85755.56319999998, 'Gaming': 24655.285099999997, 'Office': 26356.47520000001}


8.   Напишите функцию ```top_categories(orders, n=3)``` которая вернет список из первых n категорий по сумме продаж (с учетом скидок).


In [56]:
def top_categories(orders:list, n:int=3) -> list[str]:
    categories_sum = sum_by_category(orders)
    categories = sorted(categories_sum, key=categories_sum.get, reverse=True)

    return categories[:n]

print("Top categories:", top_categories(orders))

Top categories: ['Electronics', 'Accessories', 'Office']


9.   Напишите функцию ```payment_stats(orders)``` которая в качестве параметра принимает список заказов и вернет словарь в котором заказы сгруппированы по способу оплаты. Ключ - способ оплаты, значение - список заказов.


In [8]:
def payment_stats(orders: list) -> dict[str:list]:
    result = {}

    for order in orders:
        payment = order["PaymentMethod"]

        if payment not in result:
            result[payment] = []

        result[payment].append(order)

    return result

# Закомментировала строку, чтобы не перегружать экран огромным выводом
#print(payment_stats(orders))

10.   Напишите функцию ```average_discount(orders)``` которая принимает список заказов и вернет среднюю скидку по всем заказам.


In [58]:
def average_discount(orders: list) -> float:
    total_discount = 0

    for order in orders:
        total_discount += order["Discount"]

    average = total_discount / len(orders)

    return average

print("Average discount:", average_discount(orders))

Average discount: 0.14829831932773127




---



###**Task 3**. Анализ данных

In [10]:
amounts = []

for order in orders:
    quantity = order["Quantity"]
    price = float(order["UnitPrice"])
    discount = order["Discount"]

    total = quantity * price
    total = total * (1 - discount)

    amounts.append(total)

# Закомментировала строку, чтобы не перегружать экран огромным выводом
#print(amounts)

Среднее значение суммы

In [60]:
import statistics

average = statistics.mean(amounts)

print("Average sum:", average)

Average sum: 239.13360651260504


Медианное значение суммы

In [61]:
median = statistics.median(amounts)

print("Median of the sum:", median)

Median of the sum: 186.4515


Среднеквадратичное отклонение от суммы

In [64]:
deviation = statistics.stdev(amounts)

print("Standard deviation:", deviation)

Standard deviation: 197.89217483141505


Q1, Q3, IQR

In [67]:
quartiles = statistics.quantiles(amounts)

q1 = quartiles[0]
q3 = quartiles[2]

iqr = q3 - q1

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)

Q1: 74.50590000000001
Q3: 352.42494999999997
IQR: 277.91904999999997


Распределение заказов

In [68]:
difference = abs(average - median)

if difference <= median * 0.1:
    print("Order distribution is normal")
else:
    print("Order distribution is not normal")

Order distribution is not normal


Фильтрация выбросов

In [75]:
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

emissions = []

for amount in amounts:
    if amount < lower or amount > upper:
        emissions.append(amount)

print("Lower limit:", lower)
print("Upper limit:", upper)
print("Emissions:", emissions)

Lower limit: -342.37267499999996
Upper limit: 769.3035249999999
Emissions: [859.312, 785.3391, 884.9159999999999, 846.4060000000001, 899.976, 957.46, 793.6236, 864.5669999999999, 777.9839999999999, 830.648, 985.7800000000001, 967.5900000000001, 810.8, 793.716, 771.52, 947.2379999999998]


Пункты 1-6 для каждой категории товаров

In [76]:
import statistics

def product_category(orders):
    categories = {}

    for order in orders:
        category = order["Category"]
        quantity = order["Quantity"]
        price = float(order["UnitPrice"])
        discount = order["Discount"]

        total = quantity * price
        total = total * (1 - discount)

        if category not in categories:
            categories[category] = []

        categories[category].append(total)

    return categories

categories =  product_category(orders)

for category in categories:
    amounts = categories[category]

    average = statistics.mean(amounts)
    median = statistics.median(amounts)
    standard_deviation = statistics.stdev(amounts)
    quartiles = statistics.quantiles(amounts)
    q1 = quartiles[0]
    q3 = quartiles[2]
    iqr = q3 - q1

    print("Category:", category)
    print("Average:", average)
    print("Median:", median)
    print("Standard deviation:", standard_deviation)
    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)

Category: Electronics
Average: 241.08188302387268
Median: 188.96640000000002
Standard deviation: 196.55670912532625
Q1: 76.4616
Q3: 359.32050000000004
IQR: 282.85890000000006
Category: Accessories
Average: 254.4675465875371
Median: 200.3778
Standard deviation: 210.92868899910422
Q1: 75.3743
Q3: 379.69679999999994
IQR: 304.32249999999993
Category: Gaming
Average: 228.28967685185185
Median: 163.3784
Standard deviation: 194.13759467080246
Q1: 66.47197499999999
Q3: 358.84620000000007
IQR: 292.3742250000001
Category: Office
Average: 202.7421169230769
Median: 155.1932
Standard deviation: 163.97615022785314
Q1: 69.3198
Q3: 275.5809
IQR: 206.2611


Определите что выгоднее штучные заказы на большую сумму или много заказов с незначительными суммами

In [78]:
total_sales = sum(amounts)
average_order = statistics.mean(amounts)
number_of_orders = len(amounts)

print("Total sales:", total_sales)
print("Average order amount:", average_order)
print("Number of orders:", number_of_orders)

if average_order > median:
    print("Orders for larger amounts are more profitable")
else:
    print("Orders for small amounts are more profitable")

Total sales: 26356.4752
Average order amount: 202.7421169230769
Number of orders: 130
Orders for larger amounts are more profitable


Товары какой категории дают максимальные суммы

In [79]:
category_sum = sum_by_category(orders)
top_category = max(category_sum, key=category_sum.get)

print("The most profitable category:", top_category)
print("Total sales:", category_sum[top_category])

The most profitable category: Electronics
Total sales: 90887.86990000005


Какой самый популярный товар

In [80]:
popular_product = most_popular_item(orders)

print("Most Popular Item:", popular_product)

total_laptop_stand = sum(int(order["Quantity"]) for order in orders if order["Description"] == popular_product)

print(f"Number of units sold: {total_laptop_stand}")

Most Popular Item: Laptop Stand
Number of units sold: 735


Какие заказы дают большие суммы, там где пользователь идентифицирован или нет?

In [81]:
with_customer_amounts = []

for order in orders_with_customer:
    quantity = order["Quantity"]
    price = float(order["UnitPrice"])
    discount = order["Discount"]

    total = quantity * price
    total = total * (1 - discount)

    with_customer_amounts.append(total)

without_customer_amounts = []

for order in orders_without_customer:
    quantity = order["Quantity"]
    price = float(order["UnitPrice"])
    discount = order["Discount"]

    total = quantity * price
    total = total * (1 - discount)

    without_customer_amounts.append(total)

print("Average amount with CustomerID:", statistics.mean(with_customer_amounts))
print("Average amount without CustomerID:", statistics.mean(without_customer_amounts))

if with_customer_amounts > without_customer_amounts:
    print("More with CustomerID")
else:
    print("More without CustomerID")

Average amount with CustomerID: 238.0242842222222
Average amount without CustomerID: 240.12801892430278
More without CustomerID


Есть ли разница в статистиках для этих групп?

In [82]:
average_with = statistics.mean(with_customer_amounts)
median_with = statistics.median(with_customer_amounts)
stdev_with = statistics.stdev(with_customer_amounts)

average_without = statistics.mean(without_customer_amounts)
median_without = statistics.median(without_customer_amounts)
stdev_without = statistics.stdev(without_customer_amounts)

average_difference = average_without - average_with
median_difference = median_without - median_with
stdev_difference = stdev_without - stdev_with

print("Differences between the groups:")
print("Average:", round(average_difference, 2))
print("Median:", round(median_difference, 2))
print("Standard deviation:", round(stdev_difference))

print("Conclusion:")

if average_difference > 0:
    print("Without CustomerID the average orders amount is higher")
elif average_difference < 0:
    print("With CustomerID the average orders amount is higher")
else:
    print("The average orders amounts are the same")

if median_difference > 0:
    print("The median is higher for orders without CustomerID")
elif median_difference < 0:
    print("The median is higher for orders with CustomerID")
else:
    print("The medians are the same")

if stdev_difference > 0:
    print("Order amounts without CustomerID are more widely distributed")
elif stdev_difference < 0:
    print("Order amounts with CustomerID are more widely distributed")
else:
    print("The range of order amounts is the same")


Differences between the groups:
Average: 2.1
Median: 13.74
Standard deviation: 4
Conclusion:
Without CustomerID the average orders amount is higher
The median is higher for orders without CustomerID
Order amounts without CustomerID are more widely distributed


Привлекает ли людей скидка на товар?

In [84]:
discount_orders = []
no_discount_orders = []

for order in orders:
    if order["Discount"] > 0:
        discount_orders.append(order)
    else:
        no_discount_orders.append(order)

print("Discountad orders:", len(discount_orders))
print("Orders without a discount:", len(no_discount_orders))

if len(discount_orders) > len(no_discount_orders):
    print("A discount attracts customers")
else:
    print("A discount doesn't attract customers")

Discountad orders: 936
Orders without a discount: 16
A discount attracts customers




1.   Вычислите среднее значение суммы (с учетом скидки) по всем заказам.
2.   Вычислите медианное значение суммы (с учетом скидки) по всем заказам.
3.   Вычислите среднеквадратичное отклонение суммы (с учетом скидки) по всем заказам.
4.   Вычислите Q1,Q3, IQR для значения суммы (с учетом скидки) по всем заказам.
5.   Ответьте на вопрос - распределение заказов нормальное или нет.
6.   Отфильтруйте возможные выбросы в отдельный список.
7.   Выполните пункты 1- 6 для каждой категории товаров.
8.   Определите что выгоднее штучные заказы на большую сумму или много заказов с незначительными суммами.
9.   Товары какой категории дают максимальные суммы?
10.  Какой самый популярный товар?
11.  Какие заказы дают большие суммы, там где пользователь идентифицирован или нет?
12.  Есть ли разница в статистиках для этих групп?
13.  Привлекает ли людей скидка на товар?


---




###**Task 4.** Выводы
Напишите несколько предложений своих выводов и рекомендаций на основе данных продаж.


---

Выводы:
1. Скидки работают идеально. Они привлекают 98% клиентов: 936 заказов было со скидкой и всего 16 - без неё.
2. Общая выручка компании составила 227 655,19. Общее количество проданных товаров по магазину составляет 5218шт. Лидером является категория Electronics, продажи по ней принесли 90 887,87.  Самый продаваемый товар - это подставки Laptop Stand - 735 шт..
3. Среднее значение суммы заказа равно 239,13, а медианное - 186,45. Разница между ними доказала, что распределение сумм ненормальное.
4. Среднеквадратичное отклонение составляет 197,89. Это указывает, что стоимость покупок нестабильна и скачет от заказа к заказу.
5. Половина всех заказов находится в стабильных границах от 74,51 до 352,42. Межквартальный размах равен 277,92.
6. Расчет показал, что всё, что дороже 769,30, считается крупными выбросами, которые тянут среднее значение вверх. Самая большая сумма заказа - 985,78 является примером такого выброса.
7. Компании более выгодны штучные крупные заказы, а не масса мелких покупок.

Рекомендации:
1. Сформировать список клиентов, совершивших самые дорогие заказы и предложить им VIP - условия(например: персональный менеджер, закрытые распродажи, повышенный кэшбэк), чтобы удержать их.
2. Не отменять скидки, без них мы потеряем много клиентов.
3. Всегда держать подставки Laptop Stand в большом количестве, чтобы они не заканчивались.
4. Расширять ассортимент электроники и выделять больше рекламного бюджета на её продвижение, так как она приносит больше всего заказов.
5. Для поднятия чека предлагать мелкие сопутствующие товары при покупке или бесплатную доставку от определенной суммы.
6. Так как неавторизованные клиенты совершают больше половины всех заказов, их необходимо мотивировать регистрироваться. Внедрить приветственный бонус или мелкую скидку на следующий заказ за создание личного кабинета.

##**Справочные материалы.**

Для чтения csv файлов нужно:

*   Импортировать модуль csv
*   Для чтения в виде списка словарей можно использовать DictReader из модуля csv



Для получения статистических характеристик следует использовать модуль **statistics**.
Этот модуль предоставляет ряд функций:

*   **mean** - среднее значение
*   **median** - медианное значение
*   **stdev** - среднеквадратичное отклонение
*   **quantiles** - делит данные на n=4 (именованный параметр со значением по умолчанию) интервалов и возвращает n-1 точку разбиения. Если использовать значение по умолчанию то вернет список в котром будут [Q1,Q2,Q3].



**Как определяется межквартильный размах (Interquartile range) IQR?**

Межквартильный размах описывает как центральный промежуток в котором лежит 50% данных и вычисляется как
```
IQR = Q3 - Q1.
```
Q1,Q3 - верхний и нижний квартиль соответственно.

**Как сделать предположение о распределение данных?**

Если среднее значение и медианное значение близки (разница не более 10% от значения медианного), то можно говорить что распределение нормальное.

**Как определить выброс или аномалию в данных ?**

*   Нормальное распределение. Данные за пределами среднее +- 3*среднеквадратичное отклонение.
*   Распределение не является нормальным. Данные за пределами промежутка
```
Q1 - 1.5*IQR ... Q3 + 1.5*IQR
```



###**Желаю успехов в выполнении задания.**